# Checkpoint follower iMD agent

Example of an iMD agent that guides selections to match checkpoints marked in an earlier recording.

<video src="../figures/imd-checkpoint-follower.webm" controls>

## Setup runner & utilities

In [1]:
from nanover.app import OmniRunner
from nanover.openmm import OpenMMSimulation

simulation = OpenMMSimulation.from_xml_path("../systems/openmm/17-ala.openmm.zip")

OmniRunner.close_all_runners()
imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="EXAMPLE: imd checkpoint follower")
imd_runner.print_basic_info()
imd_runner.load(0)

Serving "EXAMPLE: imd checkpoint follower" (ws://localhost:61404), discoverable on all interfaces on port 54545
Available simulations:
[0]: "17-ala.openmm"
Switched to [0]: "17-ala.openmm"
Switched to [0]: "17-ala.openmm"


In [2]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)
utilities.selections.update_selection("root", renderer="cartoon")

utilities.show_logging()

Output()

## Extract checkpoints from recording

Functions to extract checkpoints from a recording made using the "mobile restraints" example:

In [3]:
from dataclasses import dataclass, field

import numpy as np
import numpy.typing as npt
from MDAnalysis import Universe

from nanover.recording.reading import RecordingEvent
from nanover.imd.imd_state import dict_to_interaction

from iter_checkpoints import iter_checkpoints


@dataclass(kw_only=True, eq=False)
class TargetGroup:
    particles: list[int] = field(default_factory=list)
    position: npt.NDArray[np.float32]
    centroid: npt.NDArray[np.float32]

    def __repr__(self):
        return f"<TargetGroup of particles {self.particles}>"


@dataclass(kw_only=True, eq=False)
class KeyFrame:
    targets: list[TargetGroup] = field(default_factory=list)


def extract_keyframes_restraints(path):
    keyframes = [restraint_keyframe_from_checkpoint(checkpoint) for checkpoint in iter_checkpoints(path)]
    keyframes = [keyframe for keyframe in keyframes if keyframe is not None]
    return keyframes


def restraint_keyframe_from_checkpoint(checkpoint: RecordingEvent):
    targets = []
    positions = checkpoint.next_frame.particle_positions

    # find active restraints at this checkpoint
    restraints = [
        dict_to_interaction(value)
        for key, value in checkpoint.next_state.items()
        if key.startswith("interaction.restraint")
    ]

    # keyframe is a target for each restraint
    for restraint in restraints:
        particles = [int(i) for i in restraint.particles]
        centroid = np.average(positions[particles], axis=0)

        target = TargetGroup(
            particles=particles,
            centroid=centroid,
            position=np.array(restraint.position),
        )
        targets.append(target)

    if not targets:
        return None

    targets.sort(key=lambda target: min(target.particles))
    return KeyFrame(targets=targets)

More aggressive method that uses every residue in each checkpoint:

In [4]:
from nanover.mdanalysis import universe_from_recording

def residue_keyframe_from_checkpoint(checkpoint: RecordingEvent, universe: Universe):
    targets = []
    positions = checkpoint.next_frame.particle_positions

    for residue in universe.residues:
        particles = [int(i) for i in residue.atoms.indices]
        centroid = np.average(positions[particles], axis=0)

        target = TargetGroup(
            particles=particles,
            centroid=centroid,
            position=centroid,
        )
        targets.append(target)

    if not targets:
        return None

    return KeyFrame(targets=targets)


def extract_keyframes_residues(path):
    universe = universe_from_recording(RECORDING_PATH)
    keyframes =  [residue_keyframe_from_checkpoint(checkpoint, universe) for checkpoint in iter_checkpoints(path)]
    keyframes = [keyframe for keyframe in keyframes if keyframe is not None]
    return keyframes

## iMD agent that follows a checkpoint

Define an iMD agent that uses a list of target centroids for particle groups and interacts to guide each group to where it should be:

In [5]:
from nanover.imd import ParticleInteraction
from nanover.trajectory import FrameData
from nanover.jupyter import ImdAgent, SceneObjectsUtility
from nanover.utilities.transforms import find_transformation_between_point_patterns, _transform_vec3s


class SmearAgent(ImdAgent):
    speed = 0.1
    keyframe: KeyFrame | None = None

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.objects = SceneObjectsUtility(self._app_server)

    def close(self, wait=False):
        super().close(wait=wait)
        self.objects.clear()

    def set_keyframe(self, keyframe: KeyFrame):
        self.interactions.clear()
        self.objects.clear()
        self.keyframe = keyframe

    def update_interactions(self, full_frame: FrameData, frame_update: FrameData):
        if self.keyframe is None:
            self.objects.clear()
            return

        keyframe_centroids = np.array([target.centroid for target in self.keyframe.targets])

        # determine current centroids of target groups
        prev_centroids = np.array([
            np.mean(full_frame.particle_positions[target.particles], axis=0)
            for target in self.keyframe.targets
        ])

        # align target centroids to current centroids
        next_centroids = _transform_vec3s(
            find_transformation_between_point_patterns(keyframe_centroids, prev_centroids),
            keyframe_centroids,
        )

        # find necessary motions
        deltas = next_centroids - prev_centroids

        # cap motions by speed
        lengths = np.linalg.norm(deltas, axis=1).reshape(-1, 1)
        cappeds = deltas / lengths
        np.clip(lengths, max=self.speed, out=lengths)
        cappeds *= lengths

        # determine final interaction positions
        target_centroids = prev_centroids + cappeds

        with (
            self.interactions as interactions,
            self.objects as objects,
        ):
            points = [target_centroids[i] for i in range(len(self.keyframe.targets))]
            points2 = [next_centroids[i] for i in range(len(self.keyframe.targets))]
            white = [1, 1, 1, .5]

            objects.update_line("backbone", positions=points, color=white, size=0.05)
            objects.update_line("backbone.target", positions=points2, color=[1, 0, 1, .5], size=0.05)

            for i, target in enumerate(self.keyframe.targets):
                error = min(1.0, np.linalg.norm(deltas, axis=1)[i])
                label = f"{error:.2g}nm"
                color = [1, 1 - error, 1 - error, 1]
                position = points[i]

                objects.update_shape(i, shape="sphere", position=position, color=color)
                # objects.update_label(i, text=label, position=position, color=color)

                if lengths[i] < 0.0001:
                    interactions.remove_interaction(f"interaction.REPLAYER.{i}")
                    continue

                interaction = ParticleInteraction(
                    particles=target.particles,
                    position=list(target_centroids[i]),
                    interaction_type="spring",
                    scale=2000,
                    max_force=10000,
                )
                interactions.update_interaction(
                    f"interaction.REPLAYER.{i}", interaction
                )


## Agent controls

In [6]:
RECORDING_PATH = "../systems/recordings/knot-tying-checkpoints.nanover.zip"
KEYFRAMES = extract_keyframes_restraints(RECORDING_PATH)
print(KEYFRAMES)

[KeyFrame(targets=[<TargetGroup of particles [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]>, <TargetGroup of particles [162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172]>]), KeyFrame(targets=[<TargetGroup of particles [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]>, <TargetGroup of particles [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]>, <TargetGroup of particles [72, 73, 74, 75, 76, 77, 78, 79, 80, 81]>, <TargetGroup of particles [102, 103, 104, 105, 106, 107, 108, 109, 110, 111]>, <TargetGroup of particles [162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172]>]), KeyFrame(targets=[<TargetGroup of particles [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]>, <TargetGroup of particles [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]>, <TargetGroup of particles [72, 73, 74, 75, 76, 77, 78, 79, 80, 81]>, <TargetGroup of particles [102, 103, 104, 105, 106, 107, 108, 109, 110, 111]>, <TargetGroup of particles [162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172]>]), KeyFrame(targets=[<TargetGroup of particles [0, 1, 2

In [7]:
KEYFRAME_INDEX = -1
AGENT = SmearAgent.from_runner(imd_runner)
AGENT.speed = .2
AGENT.start()

def smear_next():
    global KEYFRAME_INDEX
    KEYFRAME_INDEX = KEYFRAME_INDEX + 1

    if KEYFRAME_INDEX == len(KEYFRAMES):
        smear_cancel()
        return

    utilities.notify_all(f"MATCHING KEYFRAME {KEYFRAME_INDEX}")
    AGENT.set_keyframe(KEYFRAMES[KEYFRAME_INDEX])


def smear_back():
    global KEYFRAME_INDEX
    KEYFRAME_INDEX = max(0, KEYFRAME_INDEX - 1)

    utilities.notify_all(f"MATCHING KEYFRAME {KEYFRAME_INDEX}")
    AGENT.set_keyframe(KEYFRAMES[KEYFRAME_INDEX])


def smear_cancel():
    global KEYFRAME_INDEX
    KEYFRAME_INDEX = -1

    AGENT.set_keyframe(None)
    utilities.notify_all(f"RESETTING AGENT")

In [8]:
utilities.define_command("user/prev", label="prev checkpoint", handler=smear_back, icon="⏮️")
utilities.define_command("user/next", label="next checkpoint", handler=smear_next, icon="⏭️")
utilities.define_command("user/cancel", label="cancel", handler=smear_cancel, icon="⏹️")